In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from skimage.io import imread
from skimage.measure import regionprops_table
from skimage.morphology import dilation
from natsort import natsorted

from calmutils.morphology.structuring_elements import hypersphere_centered

# Nested regionprops

This recipe will perform region properties measurements given intensity images and two nested masks, for **outer and inner** objects. For example:

- **outer objects** could be nuclei
- **inner objects** could be subnuclear structures (e.g. heterochromatin clusters)

We calculate the specified region properties

- in the objects masked by the outer mask excluding the inner objects (plus a user-specified border)
- in the inner objects (ignoring those that do note lie in an outer object)

The final table is generated by joining the inner measurements with the outer on the outer label id, thus we will have one row for each inner objects, with measurements in the corresponding outer objects repeated.

In [ ]:
in_path = '/Users/david/Desktop/weihua_hp1_cluster_annotations/'

# subdirectory and file pattern of intensity images (should be single-channel TIFF)
image_subdirectory = 'images2'
image_file_pattern = '*.tif' # e.g. "*_ch0.tif" to only include images of one channel

# subdirectory & pattern for outer mask (e.g. nuclei)
segmentation_outer_subdirectory = 'segmentation_nuclei'
segmentation_outer_file_pattern = '*_cp_masks.png'

# subdirectory & pattern for inner mask (e.g. heterochromatin clusters)
segmentation_inner_subdirectory = 'segmentation_hp1'
segmentation_inner_file_pattern = '*_segmented.tif'

# where to save results
output_subdirectory = "nested_regionprops"

# border around inner objects to exclude from measurements in outer objects
# e.g. if we calculate mean intensity in the nucleus, we ignore areas of heterochromatin clusters dilated by this radius
# set to 0 to not ignore anything
border_ignore_radius = 3

properties_to_include = ('area', 'label', 'intensity_mean', 'intensity_std')

In [ ]:
# get sorted list of files
image_files = natsorted((Path(in_path) / image_subdirectory).glob(image_file_pattern))
segmentation_outer_files = natsorted((Path(in_path) / segmentation_outer_subdirectory).glob(segmentation_outer_file_pattern))
segmentation_inner_files = natsorted((Path(in_path) / segmentation_inner_subdirectory).glob(segmentation_inner_file_pattern))

# show combinations for verification
list(zip(image_files, segmentation_outer_files, segmentation_inner_files))

In [ ]:
out_path = Path(in_path) / output_subdirectory
if not out_path.exists():
    out_path.mkdir(exist_ok=True)

for idx in range(len(image_files)):

    image_file = image_files[idx]
    mask_outer_file = segmentation_outer_files[idx]
    mask_inner_file = segmentation_inner_files[idx]

    # load img, masks
    img = imread(image_file)
    mask_outer = imread(mask_outer_file)
    mask_inner = imread(mask_inner_file)

    # only consider inner objects within outer objects
    mask_inner = mask_inner * (mask_outer > 0)

    # outer mask with holes for inner objects (plus border ignore radius)
    if border_ignore_radius > 0:
        mask_outer_only = mask_outer * (dilation(mask_inner, hypersphere_centered(img.ndim, border_ignore_radius)) == 0)
    else:
        mask_outer_only = mask_outer * (mask_inner == 0)

    # map from inner object ids to the outer object that contains them
    # NOTE: in edge cases where an inner objects lies within two outer objects, this will arbitrarily pick one
    inner_to_outer_id_map = {int(k): int(v) for k,v in np.unique(np.stack([mask_inner, mask_outer], axis=-1)[mask_inner > 0].reshape(-1, 2), axis=0)}

    # regionprops table for both inner and outer(_only)
    df_inner = pd.DataFrame(regionprops_table(mask_inner, img, properties=properties_to_include))
    df_outer = pd.DataFrame(regionprops_table(mask_outer_only, img, properties=properties_to_include))

    # map inner label id to outer
    df_inner['label_outer'] = df_inner.label.map(inner_to_outer_id_map)

    # rename columns in outer
    df_outer.columns = [c + "_outer" for c in df_outer.columns]

    # join
    df_combined = pd.merge(df_inner, df_outer, on='label_outer')

    # add files
    df_combined['image_file'] = image_file
    df_combined['mask_outer_file'] = mask_outer_file
    df_combined['mask_inner_file'] = mask_inner_file

    df_combined.to_csv(out_path / (image_file.stem + "_nested_regionprops.csv"), index=None)

In [ ]:
## napari vis. of inner and outer masks (commented to not show in batch application)
## uncomment to run

# import napari

# if napari.current_viewer() is not None:
#     napari.current_viewer().close()

# viewer = napari.view_image(img)
# viewer.add_labels(mask_outer_only)
# viewer.add_labels(mask_inner)